# Tech Challenge Fase 2
## 03.6 — Gold Machine Learning

Prepara a base final de features para modelagem preditiva e clusterização.

## 1. Imports

In [0]:
import json
from pathlib import Path
from datetime import datetime

import pandas as pd
import numpy as np

## 2. Configuração

In [0]:
CONFIG_FILE_PATH = "/Volumes/workspace/default/vol_trio_drive/projetos/fiap/tech_challenge_fase2/config/config.json"

config = json.loads(
    Path(CONFIG_FILE_PATH).read_text(
        encoding="utf-8"
    )
)

BASE_PATH = Path(config["environment"]["base_path"])
SILVER_PATH = Path(config["paths"]["silver_path"])
GOLD_PATH = Path(config["paths"]["gold_path"])
LOG_PATH = Path(config["paths"]["log_path"])
CONFIG_PATH = Path(config["paths"]["config_path"])
EXECUTION_DATE = config["project"]["execution_date"]

print("BASE_PATH:", BASE_PATH)
print("SILVER_PATH:", SILVER_PATH)
print("GOLD_PATH:", GOLD_PATH)
print("EXECUTION_DATE:", EXECUTION_DATE)

## 3. Funções auxiliares

In [0]:
def ler_csv(caminho, sep=";", decimal=","):
    return pd.read_csv(
        caminho,
        sep=sep,
        decimal=decimal,
        encoding="utf-8",
        low_memory=False
    )


def converter_numero(serie):
    return (
        serie
        .astype(str)
        .str.replace("%", "", regex=False)
        .str.replace(">", "", regex=False)
        .str.replace(",", ".", regex=False)
        .str.strip()
        .replace({
            "": np.nan,
            "nan": np.nan,
            "None": np.nan,
            "<NA>": np.nan,
            "-": np.nan
        })
        .pipe(pd.to_numeric, errors="coerce")
    )


def normalizar_codigo(serie):
    return (
        serie
        .astype(str)
        .str.replace(".0", "", regex=False)
        .str.strip()
    )


def salvar_csv(df, destino, nome_arquivo):
    destino = Path(destino)
    destino.mkdir(parents=True, exist_ok=True)

    df.to_csv(
        destino / nome_arquivo,
        sep=";",
        decimal=",",
        encoding="utf-8",
        index=False
    )

## 4. Preparação da base de modelagem

In [0]:
df_ml = ler_csv(
    GOLD_PATH / "indicadores" / "GOLD_INDICADORES.csv"
)

features_candidatas = [
    "PC_ALUNO_ALFABETIZADO",
    "VL_MEDIA_LP",
    "META_FINAL_2030",
    "gap_meta_2030",
    "qtd_alunos",
    "qtd_escolas",
    "qtd_presentes_lp",
    "qtd_alfabetizados",
    "taxa_participacao_lp",
    "taxa_alfabetizacao_alunos",
    "proficiencia_media_lp"
]

for coluna in features_candidatas:
    if coluna in df_ml.columns:
        df_ml[coluna] = converter_numero(df_ml[coluna])

df_ml["risco_nao_atingir_meta"] = np.select(
    [
        df_ml["gap_meta_2030"].isna(),
        df_ml["gap_meta_2030"] > 0
    ],
    [
        pd.NA,
        1
    ],
    default=0
)

df_ml["risco_nao_atingir_meta"] = (
    df_ml["risco_nao_atingir_meta"]
    .astype("Int64")
)

colunas_obrigatorias = [
    c for c in [
        "PC_ALUNO_ALFABETIZADO",
        "VL_MEDIA_LP",
        "META_FINAL_2030",
        "gap_meta_2030",
        "qtd_alunos",
        "taxa_participacao_lp",
        "taxa_alfabetizacao_alunos",
        "proficiencia_media_lp"
    ]
    if c in df_ml.columns
]

df_ml["registro_completo_modelo"] = (
    df_ml[colunas_obrigatorias]
    .notna()
    .all(axis=1)
)

df_ml_validos = df_ml[df_ml["registro_completo_modelo"]].copy()
df_ml_incompletos = df_ml[~df_ml["registro_completo_modelo"]].copy()

print("Válidos:", len(df_ml_validos))
print("Incompletos:", len(df_ml_incompletos))

## 5. Persistência

In [0]:
salvar_csv(
    df_ml,
    GOLD_PATH / "base_modelo_ia",
    "BASE_MODELO_IA_COMPLETA.csv"
)

salvar_csv(
    df_ml_validos,
    GOLD_PATH / "base_modelo_ia",
    "BASE_MODELO_IA_VALIDOS.csv"
)

salvar_csv(
    df_ml_incompletos,
    GOLD_PATH / "base_modelo_ia",
    "BASE_MODELO_IA_INCOMPLETOS.csv"
)

print("Bases de Machine Learning salvas com sucesso.")